## Loading dataframe and defining features and target

In [1]:
from __future__ import print_function, division   # Ensures Python3 printing & division standard
import pandas as pd 
from pandas import Series, DataFrame 
from matplotlib import pyplot as plt
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.metrics import roc_curve
from sklearn.metrics import auc
from sklearn.metrics import confusion_matrix
import lightgbm as lgb
from lightgbm import early_stopping
import time
import shap
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from tqdm.notebook import tqdm
from matplotlib import pyplot as plt
import warnings
warnings.filterwarnings("ignore")

SavePlots = False

data = pd.DataFrame(np.genfromtxt('data_mice_encoded_aggr.csv', delimiter=',', names=True))

data["CLAIM_RATE"] = (
    data["CLAIM_COUNT"] / data["EXPOSURE"]
)

target = 'CLAIM_RATE'

features = [
    col for col in data.columns
    if col not in ['CLAIM_COUNT', 'CLAIM_SIZE', 'CLAIM_SIZE_INDEX', 
                   'f0', 'POLICY', 'EXPOSURE', 'OUTER_WALLS', 'HEATING_TYPE', 
                   'WATER_SUPPLY_TYPE', 'ROOF_TYPE', 'CLAIM_RATE']
]

data = data[data["EXPOSURE"] > 0]

X = data[features]
y = data[target]

feature_names = np.array(X.columns)

## Defining candidates for final feature set

In [2]:
top10_feat = ['DEDUCTIBLE',
              'RESIDENTIAL_AREA', 
              'HEATING_TYPE_CAT_District_Heating',
              'CONSTRUCTION_YEAR',
              'HOUDEN10KM',
              'HEATING_TYPE_CAT_Heat_Pump',
              'BUILDINGS',
              'BASEMENT_AREA',
              'ROOF_TYPE_CAT_Tar_Paper',
              'WATER_SUPPLY_TYPE_CAT_Private_Water_Supply']

top20_feat = top10_feat + ['WATER_SUPPLY_TYPE_CAT_Water_Extraction_Plant',
                           'ROOF_TYPE_CAT_Fiber_Cement_Asbestos',
                           'HEATING_TYPE_CAT_Central_Heating_Two_Units',
                           'CONSERVATORY_AREA',
                           'HEATING_TYPE_CAT_Stove_Fireplace',
                           'OUTER_WALLS_CAT_Brick_Walls',
                           'HEATING_TYPE_CAT_Electric_Heating',
                           'ROOF_TYPE_CAT_Concrete_Tiles',
                           'WATER_SUPPLY_TYPE_CAT_Non_Public_Water_Supply',
                           'FLOORS']

top_unique = ['DEDUCTIBLE',
              'CONSTRUCTION_YEAR',
              'BUILDINGS',
              'FLOORS',
              'WETROOMS',
              'RESIDENTIAL_AREA', 
              'BASEMENT_AREA',
              'CONSERVATORY_AREA',
              'ROOF_TYPE_CAT_Tar_Paper',
              'HEATING_TYPE_CAT_District_Heating',
              'WATER_SUPPLY_TYPE_CAT_Private_Water_Supply',           
              'HOUDEN10KM']


data10 = data[top10_feat + ["CLAIM_RATE"]]
data20 = data[top20_feat + ["CLAIM_RATE"]]
data_unique = data[top_unique + ["CLAIM_RATE"]]

print(data20.head(3))

groups = {
    "top10": data[top10_feat],
    "top20": data[top20_feat],
    "unique": data[top_unique]
}

   DEDUCTIBLE  RESIDENTIAL_AREA  HEATING_TYPE_CAT_District_Heating  \
0         0.0              84.0                                0.0   
1         0.0              89.0                                0.0   
2         0.0             122.0                                0.0   

   CONSTRUCTION_YEAR  HOUDEN10KM  HEATING_TYPE_CAT_Heat_Pump  BUILDINGS  \
0             1685.0         5.0                         0.0        1.0   
1             1685.0         3.0                         0.0        1.0   
2             1685.0         4.3                         0.0        1.0   

   BASEMENT_AREA  ROOF_TYPE_CAT_Tar_Paper  \
0            0.0                      0.0   
1            0.0                      0.0   
2            0.0                      0.0   

   WATER_SUPPLY_TYPE_CAT_Private_Water_Supply  ...  \
0                                         0.0  ...   
1                                         1.0  ...   
2                                         0.0  ...   

   ROOF_TYPE_CAT_Fib

## Initial HP grid search
### Tree-based models (LGBM, XGB, CAT)

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb

from xgboost import XGBRegressor
from catboost import CatBoostRegressor

from sklearn.model_selection import train_test_split

np.random.seed(42)

y_full = np.log1p(y.copy())

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_pred - y_true) ** 2))


LGBM_GRID = [
    {"num_leaves": 31, "learning_rate": 0.05},
    {"num_leaves": 63, "learning_rate": 0.05},
    {"num_leaves": 63, "learning_rate": 0.1},
]

XGB_GRID = [
    {"max_depth": 4, "learning_rate": 0.05},
    {"max_depth": 6, "learning_rate": 0.05},
    {"max_depth": 6, "learning_rate": 0.1},
]

CAT_GRID = [
    {"depth": 6, "learning_rate": 0.05},
    {"depth": 8, "learning_rate": 0.05},
    {"depth": 6, "learning_rate": 0.1},
]

n_seeds = 5


def build_lgbm(seed, params):
    return lgb.LGBMRegressor(
        objective="regression",
        n_estimators=5000,   
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=seed,
        verbose=-1,
        **params
    )


def build_xgb(seed, params):
    return XGBRegressor(
        objective="reg:squarederror",
        n_estimators=5000,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        eval_metric="rmse",
        random_state=seed,
        n_jobs=-1,
        **params
    )


def build_cat(seed, params):
    return CatBoostRegressor(
        loss_function="RMSE",
        iterations=5000,
        subsample=0.8,
        random_seed=seed,
        verbose=0,
        task_type="CPU",
        **params
    )


MODELS = {
    "LightGBM": (build_lgbm, LGBM_GRID),
    "XGBoost": (build_xgb, XGB_GRID),
    "CatBoost": (build_cat, CAT_GRID)
}

results = {}

for model_name, (builder, grid) in MODELS.items():

    print(f"\n\n==============================")
    print(f"MODEL: {model_name}")
    print(f"==============================")

    results[model_name] = []

    for config_id, params in enumerate(grid):

        print(f"\n>>> Config {config_id+1}/{len(grid)}: {params}")

        config_results = {}

        for group_name, X_full in groups.items():

            print(f"  -> Group: {group_name}")

            losses = []

            for seed in range(n_seeds):

                X_train, X_valid, y_train, y_valid = train_test_split(
                    X_full,
                    y_full,
                    test_size=0.25,
                    random_state=seed
                )

                model = builder(seed, params)

                if model_name == "LightGBM":
                    model.fit(
                        X_train,
                        y_train,
                        eval_set=[(X_valid, y_valid)],
                        callbacks=[lgb.early_stopping(50, verbose=False)]
                    )

                elif model_name == "XGBoost":
                    model.fit(
                        X_train,
                        y_train,
                        eval_set=[(X_valid, y_valid)],
                        verbose=False
                    )

                elif model_name == "CatBoost":
                    model.fit(
                        X_train,
                        y_train,
                        eval_set=(X_valid, y_valid),
                        early_stopping_rounds=50
                    )

                pred = model.predict(X_valid)

                loss = rmse(y_valid, pred)
                losses.append(loss)

            config_results[group_name] = np.mean(losses)

        rank_score = np.mean(list(config_results.values()))

        results[model_name].append({
            "params": params,
            "scores": config_results,
            "rank_score": rank_score
        })

rows = []

for model_name in results:
    for cfg in results[model_name]:
        params = cfg["params"]
        scores = cfg["scores"]

        for feature_set, score in scores.items():
            rows.append({
                "model": model_name,
                "feature_set": feature_set,
                "score": score,
                **params
            })

df_results = pd.DataFrame(rows)

df_sorted = df_results.sort_values(
    "score",
    ascending=True
)

print("\n===== FINAL RESULTS =====")
print(df_sorted)



MODEL: LightGBM

>>> Config 1/3: {'num_leaves': 31, 'learning_rate': 0.05}
  -> Group: top10
  -> Group: top20
  -> Group: unique

>>> Config 2/3: {'num_leaves': 63, 'learning_rate': 0.05}
  -> Group: top10
  -> Group: top20
  -> Group: unique

>>> Config 3/3: {'num_leaves': 63, 'learning_rate': 0.1}
  -> Group: top10
  -> Group: top20
  -> Group: unique


MODEL: XGBoost

>>> Config 1/3: {'max_depth': 4, 'learning_rate': 0.05}
  -> Group: top10
  -> Group: top20
  -> Group: unique

>>> Config 2/3: {'max_depth': 6, 'learning_rate': 0.05}
  -> Group: top10
  -> Group: top20
  -> Group: unique

>>> Config 3/3: {'max_depth': 6, 'learning_rate': 0.1}
  -> Group: top10
  -> Group: top20
  -> Group: unique


MODEL: CatBoost

>>> Config 1/3: {'depth': 6, 'learning_rate': 0.05}
  -> Group: top10
  -> Group: top20
  -> Group: unique

>>> Config 2/3: {'depth': 8, 'learning_rate': 0.05}
  -> Group: top10
  -> Group: top20
  -> Group: unique

>>> Config 3/3: {'depth': 6, 'learning_rate': 0.1}
  -

### NN model (MLP)

In [ ]:
import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

firedata = data.copy()
y_full = np.log1p(y.copy())

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_pred - y_true) ** 2))

MLP_GRID = [
    {
        "hidden_layer_sizes": (64, 64),
        "alpha": 1e-4,
        "learning_rate_init": 1e-3
    },
    {
        "hidden_layer_sizes": (128, 64),
        "alpha": 1e-4,
        "learning_rate_init": 1e-3
    },
    {
        "hidden_layer_sizes": (128, 128),
        "alpha": 1e-3,
        "learning_rate_init": 5e-4
    }
]

n_seeds = 10


def build_mlp(seed, params):

    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", MLPRegressor(
            max_iter=800,  # increased slightly for stability
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=20,
            random_state=seed,
            **params
        ))
    ])


results = []

print("\n==============================")
print("MLP MODEL SEARCH (IMPROVED)")
print("==============================")

for config_id, params in enumerate(MLP_GRID):

    print(f"\n>>> Config {config_id+1}/{len(MLP_GRID)}")
    print(params)

    for group_name, X_full in groups.items():

        print(f"\n  -> Feature set: {group_name}")

        losses = []

        for seed in tqdm(range(n_seeds), desc=f"Config {config_id+1} | {group_name}", leave=False):

            X_train, X_valid, y_train, y_valid = train_test_split(
                X_full,
                y_full,
                test_size=0.25,
                random_state=seed
            )

            model = build_mlp(seed, params)

            model.fit(X_train, y_train)

            pred = model.predict(X_valid)

            loss = rmse(y_valid, pred)
            losses.append(loss)

        mean_loss = np.mean(losses)

        results.append({
            "model": "MLP",
            "feature_set": group_name,
            "score": mean_loss,
            **params
        })

        print(f"     RMSE (log-space) = {mean_loss:.6f}")

df_mlp = pd.DataFrame(results)

df_mlp_sorted = (
    df_mlp
    .sort_values("score", ascending=True)
    .reset_index(drop=True)
)

print("\n===== FINAL MLP RESULTS =====")
print(df_mlp_sorted)


MLP MODEL SEARCH (IMPROVED)

>>> Config 1/3
{'hidden_layer_sizes': (64, 64), 'alpha': 0.0001, 'learning_rate_init': 0.001}

  -> Feature set: top10


Config 1 | top10:   0%|          | 0/10 [00:00<?, ?it/s]

     RMSE (log-space) = 0.071147

  -> Feature set: top20


Config 1 | top20:   0%|          | 0/10 [00:00<?, ?it/s]

     RMSE (log-space) = 0.071154

  -> Feature set: unique


...

     RMSE (log-space) = 0.071144

  -> Feature set: top20


Config 3 | top20:   0%|          | 0/10 [00:00<?, ?it/s]

     RMSE (log-space) = 0.071149

  -> Feature set: unique


Config 3 | unique:   0%|          | 0/10 [00:00<?, ?it/s]

     RMSE (log-space) = 0.071146

===== FINAL MLP RESULTS =====
  model feature_set     score hidden_layer_sizes   alpha  learning_rate_init
0   MLP       top10  0.071144         (128, 128)  0.0010              0.0005
1   MLP       top10  0.071145          (128, 64)  0.0001              0.0010
2   MLP      unique  0.071146         (128, 128)  0.0010              0.0005
3   MLP       top10  0.071147           (64, 64)  0.0001              0.0010
4   MLP       top20  0.071149         (128, 128)  0.0010              0.0005
5   MLP      unique  0.071150           (64, 64)  0.0001              0.0010
6   MLP      unique  0.071151          (128, 64)  0.0001              0.0010
7   MLP       top20  0.071152          (128, 64)  0.0001              0.0010
8   MLP       top20  0.071154           (64, 64)  0.0001              0.0010


In [4]:
df_lgbm = df_sorted[df_sorted["model"] == "LightGBM"].copy() 
df_lgbm = df_lgbm.drop(columns=["max_depth", "depth"], errors="ignore") 
print(df_lgbm.head(3))

df_xgb = df_sorted[df_sorted["model"] == "XGBoost"].copy() 
df_xgb = df_xgb.drop(columns=["num_leaves", "depth"], errors="ignore") 
print(df_xgb.head(3))

df_cat = df_sorted[df_sorted["model"] == "CatBoost"].copy() 
df_cat = df_cat.drop(columns=["max_depth", "num_leaves"], errors="ignore") 
print(df_cat.head(3))

print(df_mlp_sorted.head(3))

NameError: name 'df_sorted' is not defined

In [3]:
winner = df_mlp_sorted.sort_values("score").iloc[0]
print(winner)

NameError: name 'df_mlp_sorted' is not defined

In [3]:
import pandas as pd

winner = pd.Series({
    "model": "MLP",
    "feature_set": "top10",
    "score": 0.071144,
    "hidden_layer_sizes": (128, 128),
    "alpha": 0.001,
    "learning_rate_init": 0.0005
})

## Finetuning 

### NN (MLP) using (only) Optuna

In [ ]:
import time
import numpy as np
import pandas as pd
import optuna

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor

best_feature_set = "top10"

best_params = {
    "hidden_layer_sizes": (128, 128),
    "alpha": 0.001,
    "learning_rate_init": 0.0005
}

feature_map = {
    "top10": top10_feat,
    "top20": top20_feat,
    "unique": top_unique
}

X_full = data[feature_map[best_feature_set]].astype(np.float32)
y_full = np.log1p(data["CLAIM_RATE"].values)


n_splits = 3
MAX_ITER = 300
N_TRIALS = 25


def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_pred - y_true) ** 2))


def eval_mlp(params):

    losses = []

    for seed in range(n_splits):

        X_train, X_valid, y_train, y_valid = train_test_split(
            X_full,
            y_full,
            test_size=0.25,
            random_state=seed
        )

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("mlp", MLPRegressor(
                random_state=seed,
                max_iter=MAX_ITER,
                early_stopping=True,
                validation_fraction=0.1,
                n_iter_no_change=15,
                **params
            ))
        ])

        model.fit(X_train, y_train)

        pred = model.predict(X_valid)

        losses.append(rmse(y_valid, pred))

    return np.mean(losses)


def objective(trial):

    params = {
        "hidden_layer_sizes": trial.suggest_categorical(
            "hidden_layer_sizes",
            [
                (128, 64),
                (128, 128),
                (256, 128)
            ]
        ),

        "alpha": trial.suggest_float(
            "alpha",
            5e-4,
            5e-3,
            log=True
        ),

        "learning_rate_init": trial.suggest_float(
            "learning_rate_init",
            2e-4,
            1e-3,
            log=True
        )
    }

    return eval_mlp(params)


start = time.time()

study = optuna.create_study(direction="minimize")

study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True
)

print("\nOptuna runtime:", round(time.time() - start, 2), "sec")

print("\n===== BEST RESULT =====")
print("Score:", study.best_value)
print("Params:", study.best_params)

final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPRegressor(
        random_state=42,
        max_iter=500,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=15,
        **study.best_params
    ))
])

final_model.fit(X_full, y_full)

print("\nFinal model trained.")

[I 2026-06-04 14:43:19,685] A new study created in memory with name: no-name-858a3cab-5730-4805-902b-6eff215b1923


  0%|          | 0/25 [00:00<?, ?it/s]

[I 2026-06-04 14:45:35,947] Trial 0 finished with value: 0.07067609675824298 and parameters: {'hidden_layer_sizes': (128, 64), 'alpha': 0.001747485554184701, 'learning_rate_init': 0.0008329112740537416}. Best is trial 0 with value: 0.07067609675824298.
[I 2026-06-04 18:41:59,134] Trial 1 finished with value: 0.07067403850816895 and parameters: {'hidden_layer_sizes': (256, 128), 'alpha': 0.0036775674718708346, 'learning_rate_init': 0.00029099867956671263}. Best is trial 1 with value: 0.07067403850816895.
[I 2026-06-04 19:00:25,101] Trial 2 finished with value: 0.07067384578646065 and parameters: {'hidden_layer_sizes': (256, 128), 'alpha': 0.0006438040355269053, 'learning_rate_init': 0.0005199326479651052}. Best is trial 2 with value: 0.07067384578646065.
[I 2026-06-04 19:27:30,353] Trial 3 finished with value: 0.07067899362734946 and parameters: {'hidden_layer_sizes': (256, 128), 'alpha': 0.0009759760972881811, 'learning_rate_init': 0.0007380151116822937}. Best is trial 2 with value: 0.

#### Final validation

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import RepeatedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor

best_params = {
    "hidden_layer_sizes": (128, 128),
    "alpha": 0.002435030695326177,
    "learning_rate_init": 0.0003919806274240008
}

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_pred - y_true) ** 2))

rkf = RepeatedKFold(
    n_splits=5,
    n_repeats=2,
    random_state=42
)

scores = []

print("\n==============================")
print("FINAL REPEATED-KFOLD VALIDATION")
print("==============================")

fold = 1

for train_idx, valid_idx in rkf.split(X_full):

    X_train = X_full.iloc[train_idx]
    X_valid = X_full.iloc[valid_idx]

    y_train = y_full[train_idx]
    y_valid = y_full[valid_idx]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPRegressor(
            random_state=42,
            max_iter=500,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=20,
            **best_params
        ))
    ])

    model.fit(X_train, y_train)

    pred = model.predict(X_valid)

    score = rmse(y_valid, pred)

    scores.append(score)

    print(f"Fold {fold:2d}: RMSE (log) = {score:.6f}")

    fold += 1


scores = np.array(scores)

summary = pd.DataFrame({
    "fold": np.arange(1, len(scores) + 1),
    "score": scores
})

print("\n===== FOLD RESULTS =====")
print(summary)

mean_rmse = scores.mean()
std_rmse = scores.std(ddof=1)

ci95 = 1.96 * std_rmse / np.sqrt(len(scores))

print("\n===== FINAL ESTIMATE =====")

print(f"Mean RMSE : {mean_rmse:.6f}")
print(f"Std RMSE  : {std_rmse:.6f}")
print(f"Min RMSE  : {scores.min():.6f}")
print(f"Max RMSE  : {scores.max():.6f}")

print(
    f"95% CI    : "
    f"{mean_rmse:.6f} ± {ci95:.6f}"
)


final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPRegressor(
        random_state=42,
        max_iter=750,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        **best_params
    ))
])

final_model.fit(X_full, y_full)

print("\nFinal model fitted on full dataset.")


FINAL REPEATED-KFOLD VALIDATION
Fold  1: RMSE (log) = 0.072692
Fold  2: RMSE (log) = 0.070154
Fold  3: RMSE (log) = 0.072277
Fold  4: RMSE (log) = 0.071688
Fold  5: RMSE (log) = 0.071208
Fold  6: RMSE (log) = 0.070187
Fold  7: RMSE (log) = 0.075024
Fold  8: RMSE (log) = 0.069572
Fold  9: RMSE (log) = 0.071383
Fold 10: RMSE (log) = 0.071731

===== FOLD RESULTS =====
   fold     score
0     1  0.072692
1     2  0.070154
2     3  0.072277
3     4  0.071688
4     5  0.071208
5     6  0.070187
6     7  0.075024
7     8  0.069572
8     9  0.071383
9    10  0.071731

===== FINAL ESTIMATE =====
Mean RMSE : 0.071592
Std RMSE  : 0.001556
Min RMSE  : 0.069572
Max RMSE  : 0.075024
95% CI    : 0.071592 ± 0.000965

Final model fitted on full dataset.


### NN (MLP) using Grid, Random and Optuna

In [ ]:
import time
import numpy as np
import pandas as pd
import optuna

from sklearn.model_selection import (
    train_test_split,
    ParameterSampler
)
from scipy.stats import loguniform

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor

#winner = df_mlp_sorted.sort_values("score").iloc[0]

feature_map = {
    "top10": top10_feat,
    "top20": top20_feat,
    "unique": top_unique
}

best_feature_set = winner["feature_set"]

print("Using feature set:", best_feature_set)

X_full = data[feature_map[best_feature_set]].astype(np.float32)
y_full = np.log1p(data["CLAIM_RATE"].values)

n_seeds = 3

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_pred - y_true) ** 2))

def eval_mlp(params):

    losses = []

    for seed in range(n_seeds):

        X_train, X_valid, y_train, y_valid = train_test_split(
            X_full,
            y_full,
            test_size=0.25,
            random_state=seed
        )

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("mlp", MLPRegressor(
                random_state=seed,
                max_iter=300,
                early_stopping=True,
                validation_fraction=0.1,
                n_iter_no_change=20,
                **params
            ))
        ])

        model.fit(X_train, y_train)

        pred = model.predict(X_valid)

        losses.append(rmse(y_valid, pred))

    return np.mean(losses)

grid = {
    "hidden_layer_sizes": [
        (128, 64),
        (128, 128),
        (256, 128)
    ],
    "alpha": [
        1e-4,
        1e-3,
        1e-2
    ],
    "learning_rate_init": [
        1e-4,
        5e-4,
        1e-3
    ]
}

from itertools import product

grid_params = [
    {
        "hidden_layer_sizes": h,
        "alpha": a,
        "learning_rate_init": lr
    }
    for h, a, lr in product(
        grid["hidden_layer_sizes"],
        grid["alpha"],
        grid["learning_rate_init"]
    )
]

start = time.time()

grid_results = []

for i, params in enumerate(grid_params, start=1):

    print(f"Grid {i}/{len(grid_params)}")

    score = eval_mlp(params)

    grid_results.append({
        **params,
        "score": score
    })

grid_time = time.time() - start

df_grid = (
    pd.DataFrame(grid_results)
    .sort_values("score")
    .reset_index(drop=True)
)

random_space = {
    "hidden_layer_sizes": [
        (128, 64),
        (128, 128),
        (256, 128)
    ],

    "alpha": loguniform(
        1e-5,
        1e-1
    ),

    "learning_rate_init": loguniform(
        1e-4,
        1e-2
    )
}

random_params = list(
    ParameterSampler(
        random_space,
        n_iter=20,
        random_state=42
    )
)

start = time.time()

random_results = []

for i, params in enumerate(random_params, start=1):

    print(f"Random {i}/{len(random_params)}")

    score = eval_mlp(params)

    random_results.append({
        **params,
        "score": score
    })

random_time = time.time() - start

df_random = (
    pd.DataFrame(random_results)
    .sort_values("score")
    .reset_index(drop=True)
)

def objective(trial):

    hidden_layer_sizes = trial.suggest_categorical(
        "hidden_layer_sizes",
        [
            (128, 64),
            (128, 128),
            (256, 128)
        ]
    )

    params = {
        "hidden_layer_sizes": hidden_layer_sizes,

        "alpha": trial.suggest_float(
            "alpha",
            1e-5,
            1e-1,
            log=True
        ),

        "learning_rate_init": trial.suggest_float(
            "learning_rate_init",
            1e-4,
            1e-2,
            log=True
        )
    }

    return eval_mlp(params)


start = time.time()

study = optuna.create_study(
    direction="minimize"
)

study.optimize(
    objective,
    n_trials=20,
    show_progress_bar=True
)

optuna_time = time.time() - start

comparison = pd.DataFrame([
    {
        "method": "Grid",
        "best_score": df_grid["score"].min(),
        "runtime_sec": grid_time
    },
    {
        "method": "Random",
        "best_score": df_random["score"].min(),
        "runtime_sec": random_time
    },
    {
        "method": "Optuna",
        "best_score": study.best_value,
        "runtime_sec": optuna_time
    }
]).sort_values("best_score")

print("\n===== FINAL COMPARISON =====")
print(comparison)

print("\n===== BEST GRID PARAMS =====")
print(df_grid.iloc[0])

print("\n===== BEST RANDOM PARAMS =====")
print(df_random.iloc[0])

print("\n===== BEST OPTUNA PARAMS =====")
print(study.best_params)

best_params = study.best_params

final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPRegressor(
        random_state=42,
        max_iter=300,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        **best_params
    ))
])

final_model.fit(X_full, y_full)

print("\nFinal model trained.")

Using feature set: top10
Grid 1/27
Grid 2/27


### Tree (CAT) using Grid, Random and Optuna

In [ ]:
import time
import numpy as np
import pandas as pd
import optuna

from sklearn.model_selection import train_test_split, ParameterSampler
from scipy.stats import randint, loguniform
from sklearn.neural_network import MLPRegressor

winner = df_mlp_sorted.sort_values("score").iloc[0]

feature_map = {
    "top10": top10_feat,
    "top20": top20_feat,
    "unique": top_unique
}

best_feature_set = winner["feature_set"]

X_full = data[feature_map[best_feature_set]].astype(np.float32)

y_full = np.log1p(data["CLAIM_RATE"].values)

n_seeds = 3

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_pred - y_true) ** 2))

def eval_mlp(params):

    losses = []

    for seed in range(n_seeds):

        X_train, X_valid, y_train, y_valid = train_test_split(
            X_full,
            y_full,
            test_size=0.25,
            random_state=seed
        )

        model = CatBoostRegressor(
            iterations=3000,
            loss_function="RMSE",
            random_seed=seed,
            verbose=0,
            early_stopping_rounds=50,
            **params
        )

        model.fit(
            X_train,
            y_train,
            eval_set=(X_valid, y_valid)
        )

        pred = model.predict(X_valid)

        losses.append(rmse(y_valid, pred))

    return np.mean(losses)


#Grid search

grid = {
    "depth": [4, 5, 6, 7, 8],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "l2_leaf_reg": [1, 3, 5, 10]
}

grid_params = [
    dict(zip(grid.keys(), v))
    for v in np.array(np.meshgrid(*grid.values())).T.reshape(-1, len(grid))
]

start = time.time()

grid_results = []

for i, params in enumerate(grid_params, start=1):

    print(f"Grid {i}/{len(grid_params)}")

    score = eval_mlp(params)

    grid_results.append({
        **params,
        "score": score
    })

grid_time = time.time() - start

df_grid = (
    pd.DataFrame(grid_results)
    .sort_values("score")
    .reset_index(drop=True)
)


#Randon search

random_space = {
    "depth": randint(4, 10),
    "learning_rate": loguniform(0.01, 0.2),
    "l2_leaf_reg": randint(1, 20)
}

random_params = list(
    ParameterSampler(
        random_space,
        n_iter=20,
        random_state=42
    )
)

start = time.time()

random_results = []

for i, params in enumerate(random_params, start=1):

    print(f"Random {i}/{len(random_params)}")

    score = eval_mlp(params)

    random_results.append({
        **params,
        "score": score
    })

random_time = time.time() - start

df_random = (
    pd.DataFrame(random_results)
    .sort_values("score")
    .reset_index(drop=True)
)


#Optuna search

def objective(trial):

    params = {
        "depth": trial.suggest_int("depth", 4, 10),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.2,
            log=True
        ),

        "l2_leaf_reg": trial.suggest_int("l2_leaf_reg", 1, 20)
    }

    return eval_mlp(params)


start = time.time()

study = optuna.create_study(direction="minimize")

study.optimize(
    objective,
    n_trials=20,
    show_progress_bar=True
)

optuna_time = time.time() - start



comparison = pd.DataFrame([
    {
        "method": "Grid",
        "best_score": df_grid["score"].min(),
        "runtime_sec": grid_time
    },
    {
        "method": "Random",
        "best_score": df_random["score"].min(),
        "runtime_sec": random_time
    },
    {
        "method": "Optuna",
        "best_score": study.best_value,
        "runtime_sec": optuna_time
    }
]).sort_values("best_score")


print("\n===== FINAL COMPARISON =====")
print(comparison)

print("\n===== BEST GRID PARAMS =====")
print(df_grid.iloc[0])

print("\n===== BEST RANDOM PARAMS =====")
print(df_random.iloc[0])

print("\n===== BEST OPTUNA PARAMS =====")
print(study.best_params)

Grid 1/80
Grid 2/80
Grid 3/80
Grid 4/80
Grid 5/80
Grid 6/80
Grid 7/80
Grid 8/80
Grid 9/80
Grid 10/80
Grid 11/80
Grid 12/80
Grid 13/80
Grid 14/80
Grid 15/80
Grid 16/80
Grid 17/80
Grid 18/80
Grid 19/80
Grid 20/80
Grid 21/80
Grid 22/80
Grid 23/80
Grid 24/80
Grid 25/80
Grid 26/80
Grid 27/80
Grid 28/80
Grid 29/80
Grid 30/80
Grid 31/80
Grid 32/80
Grid 33/80
Grid 34/80
Grid 35/80
Grid 36/80
Grid 37/80
Grid 38/80
Grid 39/80
Grid 40/80
Grid 41/80
Grid 42/80
Grid 43/80
Grid 44/80
Grid 45/80
Grid 46/80
Grid 47/80
Grid 48/80
Grid 49/80
Grid 50/80
Grid 51/80
Grid 52/80
Grid 53/80
Grid 54/80
Grid 55/80
Grid 56/80
Grid 57/80
Grid 58/80
Grid 59/80
Grid 60/80
Grid 61/80
Grid 62/80
Grid 63/80
Grid 64/80
Grid 65/80
Grid 66/80
Grid 67/80
Grid 68/80
Grid 69/80
Grid 70/80
Grid 71/80
Grid 72/80
Grid 73/80
Grid 74/80
Grid 75/80
Grid 76/80
Grid 77/80
Grid 78/80
Grid 79/80
Grid 80/80
Random 1/20
Random 2/20
Random 3/20
Random 4/20
Random 5/20
Random 6/20
Random 7/20
Random 8/20
Random 9/20
Random 10/20
Random 1

[I 2026-06-03 18:00:45,486] A new study created in memory with name: no-name-2d82a0ba-48b1-43fe-9fe6-25bc5be34e99


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-06-03 18:00:51,059] Trial 0 finished with value: 0.07066707318803157 and parameters: {'depth': 5, 'learning_rate': 0.026409672585763905, 'l2_leaf_reg': 15}. Best is trial 0 with value: 0.07066707318803157.
[I 2026-06-03 18:01:00,841] Trial 1 finished with value: 0.07066816251138684 and parameters: {'depth': 6, 'learning_rate': 0.011602940711585654, 'l2_leaf_reg': 3}. Best is trial 0 with value: 0.07066707318803157.
[I 2026-06-03 18:01:04,340] Trial 2 finished with value: 0.07066484253034326 and parameters: {'depth': 9, 'learning_rate': 0.09060826496333306, 'l2_leaf_reg': 7}. Best is trial 2 with value: 0.07066484253034326.
[I 2026-06-03 18:01:16,630] Trial 3 finished with value: 0.07066541824978995 and parameters: {'depth': 10, 'learning_rate': 0.017614740710104194, 'l2_leaf_reg': 3}. Best is trial 2 with value: 0.07066484253034326.
[I 2026-06-03 18:01:24,034] Trial 4 finished with value: 0.07067050046879725 and parameters: {'depth': 5, 'learning_rate': 0.011066235444845595, 'l

#### Final validation

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor

best_params = study.best_params

n_seeds = 10
seed_scores = []

print("\n==============================")
print("FINAL ROBUST VALIDATION")
print("==============================")

print("\nBest parameters:")
print(best_params)


def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_pred - y_true) ** 2))


for seed in range(n_seeds):

    X_train, X_valid, y_train, y_valid = train_test_split(
        X_full,
        y_full,
        test_size=0.25,
        random_state=seed
    )

    model = CatBoostRegressor(
        iterations=3000,
        loss_function="RMSE",
        random_seed=seed,
        verbose=0,
        early_stopping_rounds=50,
        **best_params
    )

    model.fit(
        X_train,
        y_train,
        eval_set=(X_valid, y_valid)
    )

    pred = model.predict(X_valid)

    score = rmse(y_valid, pred)
    seed_scores.append(score)

    print(f"Seed {seed:2d}: RMSE (log) = {score:.6f}")


seed_scores = np.array(seed_scores)

summary = pd.DataFrame({
    "seed": np.arange(n_seeds),
    "score": seed_scores
})

print("\n===== SEED RESULTS =====")
print(summary)

print("\n===== FINAL ESTIMATE =====")

print(f"Mean RMSE : {seed_scores.mean():.6f}")
print(f"Std RMSE  : {seed_scores.std(ddof=1):.6f}")
print(f"Min RMSE  : {seed_scores.min():.6f}")
print(f"Max RMSE  : {seed_scores.max():.6f}")


final_model = CatBoostRegressor(
    iterations=int(1.2 * 3000),
    loss_function="RMSE",
    random_seed=42,
    verbose=0,
    **best_params
)

final_model.fit(X_full, y_full)

print("\nFinal model fitted on full dataset.")


FINAL ROBUST VALIDATION

Best parameters:
{'depth': 10, 'learning_rate': 0.02288511407141699, 'l2_leaf_reg': 16}
Seed  0: RMSE (log) = 0.070438
Seed  1: RMSE (log) = 0.070980
Seed  2: RMSE (log) = 0.070575
Seed  3: RMSE (log) = 0.070984
Seed  4: RMSE (log) = 0.073890
Seed  5: RMSE (log) = 0.070592
Seed  6: RMSE (log) = 0.070614
Seed  7: RMSE (log) = 0.072178
Seed  8: RMSE (log) = 0.070275
Seed  9: RMSE (log) = 0.070845

===== SEED RESULTS =====
   seed     score
0     0  0.070438
1     1  0.070980
2     2  0.070575
3     3  0.070984
4     4  0.073890
5     5  0.070592
6     6  0.070614
7     7  0.072178
8     8  0.070275
9     9  0.070845

===== FINAL ESTIMATE =====
Mean RMSE : 0.071137
Std RMSE  : 0.001101
Min RMSE  : 0.070275
Max RMSE  : 0.073890

Final model fitted on full dataset.
